## Lesson Overview

**What this lesson teaches:** how to combine semantic vector search with BM25 keyword search to create a hybrid retriever—the final retrieval component in this RAG pipeline.

**What's happening under the hood:**
1. Chunk the report and attach metadata.
2. Add every chunk to a vector index and a BM25 index.
3. Ask both indexes the same question.
4. Merge their rankings with reciprocal rank fusion (RRF).
5. Inspect the best chunks that would become context for a model.

**By the end:** you will have a hybrid retriever that handles both semantic questions and exact terms such as names, codes, and identifiers.

# Lesson 14: Hybrid Retrieval with BM25 and Vector Search

Vector search understands meaning, while keyword search rewards exact lexical matches. Hybrid retrieval combines those complementary signals instead of forcing one method to solve every kind of query.

## The Completed Retrieval Pipeline

```text
                         ┌→ vector index → semantic ranking ─┐
Document → chunks → index│                                  ├→ RRF → top context
                         └→ BM25 index  → keyword ranking ──┘

Question ───────────────────────→ search both indexes
```

The retrieved context can now be inserted into a prompt for answer generation. This lesson focuses on retrieval quality: generation cannot recover evidence that retrieval failed to find.

## Setup

Install the dependencies once if needed:

```python
%pip install voyageai python-dotenv
```

Add `VOYAGE_API_KEY=...` to a `.env` file. The setup checks both the current folder and `Claude_API_Training`. Live indexing uses the Voyage API and may incur a small charge; the sanity checks later in the notebook run locally.

In [1]:
import math
import os
import re
from collections import Counter
from pathlib import Path

try:
    import voyageai
except ImportError:
    voyageai = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

env_candidates = (Path('.env'), Path('Claude_API_Training/.env'))
env_path = next((path for path in env_candidates if path.exists()), None)
if env_path is not None:
    load_dotenv(env_path)

embedding_model = 'voyage-3-large'
api_key = os.getenv('VOYAGE_API_KEY')
client = voyageai.Client(api_key=api_key) if voyageai and api_key else None

if client is None:
    print('Setup incomplete: install voyageai and configure VOYAGE_API_KEY.')
else:
    print(f'Voyage client ready; model: {embedding_model}')

Voyage client ready; model: voyage-3-large


## Load and Chunk the Report

The structure-aware chunker preserves each Markdown heading. We also attach a stable `chunk_id` and source path so retrieved text can be traced back to its origin.

In [2]:
def chunk_by_section(document_text):
    return [
        section.strip()
        for section in re.split(r'(?=^##\s)', document_text, flags=re.MULTILINE)
        if section.strip()
    ]

report_candidates = (Path('report.md'), Path('Claude_API_Training/report.md'))
report_path = next((path for path in report_candidates if path.exists()), None)
if report_path is None:
    raise FileNotFoundError('Could not find report.md')

text = report_path.read_text(encoding='utf-8')
chunks = chunk_by_section(text)
documents = [
    {'content': chunk, 'chunk_id': index, 'source': str(report_path)}
    for index, chunk in enumerate(chunks)
]
print(f'Created {len(documents)} documents from {report_path}')

Created 15 documents from report.md


## Voyage Embedding Helper

The same model embeds both sides, but Voyage receives a different input type for stored content and user queries. Passing all document texts at once batches the indexing request.

In [3]:
def generate_embeddings(texts, input_type, model=embedding_model):
    if client is None:
        raise RuntimeError('Complete the Voyage setup before requesting embeddings.')
    if input_type not in {'document', 'query'}:
        raise ValueError("input_type must be 'document' or 'query'")

    is_single_text = isinstance(texts, str)
    batch = [texts] if is_single_text else list(texts)
    result = client.embed(batch, model=model, input_type=input_type)
    return result.embeddings[0] if is_single_text else result.embeddings

## Semantic Search: `VectorIndex`

The vector index retrieves conceptually similar chunks. It batches document embeddings when content is added, embeds each search string as a query, and ranks documents by cosine distance. Smaller distances are better.

In [4]:
class VectorIndex:
    def __init__(self, embedding_fn):
        self.embedding_fn = embedding_fn
        self.vectors = []
        self.documents = []
        self.vector_dim = None

    def add_documents(self, documents):
        if not documents:
            return
        contents = [document['content'] for document in documents]
        vectors = self.embedding_fn(contents, input_type='document')
        if len(vectors) != len(documents):
            raise ValueError('embedding count must match document count')
        for vector, document in zip(vectors, documents):
            if self.vector_dim is None:
                self.vector_dim = len(vector)
            elif len(vector) != self.vector_dim:
                raise ValueError('all vectors must have the same dimensions')
            self.vectors.append(list(vector))
            self.documents.append(document)

    @staticmethod
    def cosine_distance(vector_a, vector_b):
        dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
        magnitude_a = math.sqrt(sum(a * a for a in vector_a))
        magnitude_b = math.sqrt(sum(b * b for b in vector_b))
        if magnitude_a == 0 or magnitude_b == 0:
            return 1.0
        similarity = dot_product / (magnitude_a * magnitude_b)
        return 1.0 - max(-1.0, min(1.0, similarity))

    def search(self, query, k=5):
        if not self.vectors:
            return []
        query_vector = self.embedding_fn(query, input_type='query')
        if len(query_vector) != self.vector_dim:
            raise ValueError('query and document vectors must have the same dimensions')
        ranked = sorted(
            [
                (document, self.cosine_distance(query_vector, vector))
                for vector, document in zip(self.vectors, self.documents)
            ],
            key=lambda item: item[1],
        )
        return ranked[:k]

## Keyword Search: `BM25Index`

BM25 scores exact token matches while adjusting for term rarity and document length. Rare identifiers can be highly informative, so BM25 often catches evidence that a semantic representation may underweight. Here, higher raw BM25 scores are better.

In [5]:
class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.documents = []
        self.corpus_tokens = []
        self.document_frequencies = Counter()
        self.average_document_length = 0.0

    @staticmethod
    def tokenize(text):
        return [token for token in re.split(r'\W+', text.lower()) if token]

    def add_documents(self, documents):
        for document in documents:
            tokens = self.tokenize(document['content'])
            self.documents.append(document)
            self.corpus_tokens.append(tokens)
            self.document_frequencies.update(set(tokens))
        if self.corpus_tokens:
            self.average_document_length = (
                sum(len(tokens) for tokens in self.corpus_tokens)
                / len(self.corpus_tokens)
            )

    def score(self, query_tokens, document_tokens):
        term_counts = Counter(document_tokens)
        document_count = len(self.documents)
        document_length = len(document_tokens)
        total = 0.0
        for token in query_tokens:
            frequency = term_counts[token]
            if frequency == 0:
                continue
            containing_documents = self.document_frequencies[token]
            idf = math.log(1 + (document_count - containing_documents + 0.5)
                            / (containing_documents + 0.5))
            length_adjustment = self.k1 * (
                1 - self.b
                + self.b * document_length / self.average_document_length
            )
            total += idf * frequency * (self.k1 + 1) / (frequency + length_adjustment)
        return total

    def search(self, query, k=5):
        if not self.documents:
            return []
        query_tokens = self.tokenize(query)
        ranked = sorted(
            [
                (document, self.score(query_tokens, tokens))
                for document, tokens in zip(self.documents, self.corpus_tokens)
            ],
            key=lambda item: item[1],
            reverse=True,
        )
        return [(document, score) for document, score in ranked[:k] if score > 0]

## Combine Rankings with Reciprocal Rank Fusion

Vector distance and BM25 score use incompatible scales, so adding their raw values would be misleading. RRF ignores those values and uses rank positions instead. A document receives `1 / (k_rrf + rank)` from every list in which it appears.

Documents ranked highly by both methods accumulate the strongest fused score. Unlike distance, a **higher RRF score is better**.

In [6]:
class HybridRetriever:
    def __init__(self, *indexes):
        if not indexes:
            raise ValueError('provide at least one search index')
        self.indexes = indexes

    def add_documents(self, documents):
        for index in self.indexes:
            index.add_documents(documents)

    def search(self, query, k=3, candidates_per_index=5, k_rrf=60):
        if k <= 0 or candidates_per_index <= 0:
            raise ValueError('k and candidates_per_index must be positive')
        fused = {}
        for index in self.indexes:
            for rank, (document, _) in enumerate(
                index.search(query, k=candidates_per_index), start=1
            ):
                chunk_id = document['chunk_id']
                if chunk_id not in fused:
                    fused[chunk_id] = {'document': document, 'rrf_score': 0.0}
                fused[chunk_id]['rrf_score'] += 1 / (k_rrf + rank)

        return sorted(
            fused.values(),
            key=lambda result: result['rrf_score'],
            reverse=True,
        )[:k]

## Build the Hybrid Index

The same document objects go into both indexes. The vector side makes one batched API request, while BM25 builds its statistics locally. Keeping the same `chunk_id` lets RRF recognize a chunk returned by both systems.

In [7]:
retriever = None
if client is None:
    print('Skipping live vector indexing. Complete the setup, then rerun this cell.')
else:
    vector_index = VectorIndex(generate_embeddings)
    bm25_index = BM25Index()
    retriever = HybridRetriever(vector_index, bm25_index)
    retriever.add_documents(documents)
    print(f'Indexed {len(documents)} documents in both search systems')

Indexed 15 documents in both search systems


## Search and Inspect the Final Context

This question benefits from semantic understanding, while an error-code query would benefit strongly from BM25. The hybrid retriever collects candidates from both and returns the three highest fused ranks.

In [10]:
question = 'What is Material Compositve XT-5?'

if retriever is None:
    results = []
    print('No hybrid index yet. Run the live indexing cell first.')
else:
    results = retriever.search(question, k=3)
    for rank, result in enumerate(results, start=1):
        document = result['document']
        heading = document['content'].splitlines()[0]
        print(f"{rank}. RRF={result['rrf_score']:.5f} | {heading}")
        print(document['content'])
        print()

1. RRF=0.03279 | ## Section 4: Scientific Experimentation - Characterization of Material Composite XT-5
## Section 4: Scientific Experimentation - Characterization of Material Composite XT-5

The materials science team completed the initial characterization phase for Material Composite XT-5, a novel polymer-matrix composite developed in-house (Lab Ref: MSC-XT5-Batch007). Extensive testing focused on mechanical and thermal properties critical for potential next-generation applications. Results indicate a superior tensile strength averaging 450 ± 15 MPa, exceeding the benchmark material by approximately 18%. Thermal conductivity was measured at 0.8 ± 0.05 W/(m·K), suggesting suitability for applications requiring effective thermal management. However, preliminary fatigue testing (Cycle Count: 10^5 cycles, Stress Level: 200 MPa) revealed micro-fracturing patterns requiring further investigation. This approach to rigorous characterization is vital before considering integration into design

## Assemble the Context Block

Retrieval ends by joining the selected chunks into a context block. A generation step would place this text beside the question and instruct a model to answer only from the supplied evidence.

In [ ]:
if not results:
    context = ''
    print('No retrieved context to assemble yet.')
else:
    context = '\n\n---\n\n'.join(
        result['document']['content'] for result in results
    )
    print(context)

## Local Sanity Checks

These checks make no API calls. They verify BM25 exact-term retrieval and confirm that RRF rewards a document that ranks well in more than one result list.

In [ ]:
test_documents = [
    {'chunk_id': 0, 'content': 'The server reported ERR_MEM_ALLOC_FAIL_0x8007000E.'},
    {'chunk_id': 1, 'content': 'The security team contained the incident.'},
    {'chunk_id': 2, 'content': 'Quarterly sales increased.'},
]
test_bm25 = BM25Index()
test_bm25.add_documents(test_documents)
assert test_bm25.search('ERR_MEM_ALLOC_FAIL_0x8007000E', k=3)[0][0]['chunk_id'] == 0

class FixedIndex:
    def __init__(self, ranked_documents):
        self.ranked_documents = ranked_documents
    def add_documents(self, documents):
        pass
    def search(self, query, k=5):
        return [(document, 0.0) for document in self.ranked_documents[:k]]

test_hybrid = HybridRetriever(
    FixedIndex([test_documents[0], test_documents[1]]),
    FixedIndex([test_documents[1], test_documents[2]]),
)
fused_results = test_hybrid.search('test', k=3, candidates_per_index=2)
assert fused_results[0]['document']['chunk_id'] == 1
assert fused_results[0]['rrf_score'] > fused_results[1]['rrf_score']
assert chunks and all(document['content'] for document in documents)
print('All local checks passed.')

## Practice: Evaluate Hybrid Retrieval

Try these one at a time and predict the results before running:

1. **Exact identifier:** search for `ERR_MEM_ALLOC_FAIL_0x8007000E`. Compare the vector, BM25, and fused rankings.
2. **Semantic wording:** ask about containing an intrusion without using the report's exact words. Which index contributes the strongest candidate?
3. **Ablation test:** build a retriever with only BM25, then only the vector index. Which questions expose each method's weakness?
4. **Candidate depth:** change `candidates_per_index` from 2 to 10. When does a larger candidate pool change the final top three?
5. **Context budget:** retrieve five chunks instead of three. Decide whether the extra evidence is useful or distracting.
6. **Grounding prompt:** create a prompt containing `context` and `question`, with an instruction to say when the context lacks the answer.

## Summary

- Vector search captures semantic similarity; BM25 captures exact lexical overlap.
- Hybrid retrieval makes the system more robust across different query types.
- Reciprocal rank fusion combines rank positions without mixing incompatible raw scores.
- Stable document identifiers and metadata keep results aligned and traceable.
- Retrieved chunks should be inspected and evaluated before they become model context.
- The complete RAG flow is now: chunk, embed, index, retrieve, assemble context, and generate a grounded answer.